# Talent, Position Weights, And Economic Value

This notebook explores whether the economic target can be approximated as:

```text
economic_value ~= talent_score * position_weight
```

The goal is not to choose a final model here. The goal is to quickly test whether a talent-first model, such as one predicting `Peak_Overall`, plus learned positional conversion weights can explain much of the economic target used for draft-board decisions.

Primary questions:

- How strongly does `Peak_Overall` explain positive economic value?
- Do positions have meaningfully different talent-to-economic-value conversion rates?
- Does a simple `talent * position_weight` score produce useful draft-class rankings?
- Where does the talent-weighted board diverge from the raw economic target?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")


## Load Processed Data

This uses the current materialized feature store. The notebook expects to be run from the repository root or from the `notebooks/` directory.

In [ ]:
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

features_path = repo_root / "fof8-ml" / "data" / "processed" / "features.parquet"
features_path

In [ ]:
raw = pl.read_parquet(features_path)

required = [
    "Player_ID",
    "Year",
    "Position_Group",
    "Position",
    "Peak_Overall",
    "Career_Merit_Cap_Share",
    "DPO",
    "Career_Games_Played",
]
available = [c for c in required if c in raw.columns]

df = raw.select(available).with_columns(
    pl.col("Position_Group").cast(pl.Utf8),
    pl.col("Career_Merit_Cap_Share").clip(lower_bound=0).alias("positive_econ"),
    pl.col("DPO").clip(lower_bound=0).alias("positive_dpo"),
    pl.col("Career_Merit_Cap_Share").clip(lower_bound=0).log1p().alias("log_positive_econ"),
    (pl.col("Career_Merit_Cap_Share").clip(lower_bound=0) > 0).alias("is_positive_econ"),
)

df.head()


In [ ]:
df[["Peak_Overall", "Career_Merit_Cap_Share", "positive_econ", "DPO", "Career_Games_Played"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

## Position-Level Summary

This table gives a quick read on which positions convert observed peak talent into economic value at higher rates.

In [ ]:
position_summary = (
    df.group_by("Position_Group")
    .agg(
        pl.len().alias("n"),
        pl.col("is_positive_econ").mean().alias("positive_rate"),
        pl.col("Peak_Overall").mean().alias("mean_peak"),
        pl.col("positive_econ").mean().alias("mean_econ"),
        pl.col("positive_econ").quantile(0.95).alias("p95_econ"),
        pl.col("Career_Games_Played").mean().alias("mean_games"),
    )
    .with_columns(
        pl.when(pl.col("mean_peak") != 0)
        .then(pl.col("mean_econ") / pl.col("mean_peak"))
        .otherwise(None)
        .alias("econ_per_peak")
    )
    .sort("mean_econ", descending=True)
)
position_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

position_summary_pd = position_summary.to_pandas()

sns.barplot(
    data=position_summary_pd.sort_values("mean_econ", ascending=False),
    x="Position_Group",
    y="mean_econ",
    ax=axes[0],
)
axes[0].set_title("Mean Positive Economic Value By Position")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(
    data=position_summary_pd.sort_values("econ_per_peak", ascending=False),
    x="Position_Group",
    y="econ_per_peak",
    ax=axes[1],
)
axes[1].set_title("Mean Economic Value Per Peak Overall Point")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()


## Talent vs Economic Value

Use `log1p` for visualization because the economic target is heavily right-skewed.

In [ ]:
plot_df = df.sample(n=min(df.height, 25_000), seed=42).to_pandas()

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=plot_df,
    x="Peak_Overall",
    y="log_positive_econ",
    hue="Position_Group",
    alpha=0.35,
    s=18,
    linewidth=0,
)
plt.title("Peak Overall vs log1p(Positive Career Merit Cap Share)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Position")
plt.tight_layout()


In [ ]:
plot_data = df.filter(pl.col("is_positive_econ"))

# Compute per-position linear R2 values for facet titles.
facet_r2 = {}
for pos in plot_data.get_column("Position_Group").unique().to_list():
    group = plot_data.filter(pl.col("Position_Group") == pos)
    if group.height < 2:
        facet_r2[pos] = np.nan
        continue
    X = group.select("Peak_Overall").to_numpy()
    y = group.get_column("log_positive_econ").to_numpy()
    model = LinearRegression().fit(X, y)
    facet_r2[pos] = model.score(X, y)

g = sns.lmplot(
    data=plot_data.to_pandas(),
    x="Peak_Overall",
    y="log_positive_econ",
    col="Position_Group",
    col_wrap=4,
    scatter_kws={"alpha": 0.2, "s": 10},
    line_kws={"color": "black"},
    height=3,
)

for ax in g.axes.flat:
    pos = ax.get_title().split(" = ")[-1]
    r2 = facet_r2.get(pos)
    r2_label = "NA" if r2 is None or np.isnan(r2) else f"{r2:.2f}"
    ax.set_title(f"{pos} | R2={r2_label}")

g.fig.suptitle("Position-Specific Talent To Economic Conversion", y=1.02)


## Simple Baseline Regressions

These baselines test whether economic value is mostly talent, position, or talent-by-position interaction.

Interpretation:

- `peak_only`: a universal talent-to-economic conversion.
- `pos_only`: position priors without talent.
- `peak_plus_pos`: one universal talent slope plus position intercepts.
- `peak_x_pos`: position-specific talent slopes through zero.
- `peak_plus_pos_plus_interaction`: position intercepts plus position-specific slopes.

In [ ]:
def design_matrix(data: pl.DataFrame, mode: str, position_columns=None):
    if position_columns is None:
        position_columns = sorted(data.get_column("Position_Group").unique().to_list())

    peak = data.select("Peak_Overall").to_numpy()
    position_values = data.get_column("Position_Group").to_numpy()
    pos = np.column_stack(
        [(position_values == col).astype(float) for col in position_columns]
    )

    if mode == "peak_only":
        return peak, position_columns
    if mode == "pos_only":
        return pos, position_columns
    if mode == "peak_plus_pos":
        return np.column_stack([peak, pos]), position_columns
    if mode == "peak_x_pos":
        return pos * peak, position_columns
    if mode == "peak_plus_pos_plus_interaction":
        return np.column_stack([peak, pos, pos * peak]), position_columns

    raise ValueError(f"Unknown mode: {mode}")


def evaluate_baselines(data: pl.DataFrame, target_col: str) -> pl.DataFrame:
    modes = [
        "peak_only",
        "pos_only",
        "peak_plus_pos",
        "peak_x_pos",
        "peak_plus_pos_plus_interaction",
    ]
    rows = []
    position_columns = sorted(data.get_column("Position_Group").unique().to_list())
    y = data.get_column(target_col).to_numpy()

    for mode in modes:
        X, _ = design_matrix(data, mode, position_columns)
        model = LinearRegression().fit(X, y)
        pred = model.predict(X)
        rows.append(
            {
                "target": target_col,
                "subset_n": data.height,
                "model": mode,
                "r2": r2_score(y, pred),
                "mae": mean_absolute_error(y, pred),
            }
        )

    return pl.DataFrame(rows)


In [ ]:
baseline_results = pl.concat(
    [
        evaluate_baselines(df, "positive_econ").with_columns(pl.lit("all").alias("subset")),
        evaluate_baselines(df, "log_positive_econ").with_columns(pl.lit("all").alias("subset")),
        evaluate_baselines(df.filter(pl.col("is_positive_econ")), "positive_econ").with_columns(
            pl.lit("positive_only").alias("subset")
        ),
        evaluate_baselines(df.filter(pl.col("is_positive_econ")), "log_positive_econ").with_columns(
            pl.lit("positive_only").alias("subset")
        ),
    ],
    how="vertical_relaxed",
).sort(by=["subset", "target", "r2"], descending=[False, False, True])

baseline_results


## Learn Position Conversion Weights

Fit a separate simple line per position:

```text
log1p(positive_economic_value) ~= intercept_position + slope_position * Peak_Overall
```

The slope is a rough estimate of how much economic value each point of peak talent converts into for that position.

In [ ]:
def fit_position_slopes(data: pl.DataFrame) -> pl.DataFrame:
    rows = []
    for pos in data.get_column("Position_Group").unique().to_list():
        group = data.filter(pl.col("Position_Group") == pos)
        if group.height < 20:
            continue
        X = group.select("Peak_Overall").to_numpy()
        y = group.get_column("log_positive_econ").to_numpy()
        model = LinearRegression().fit(X, y)
        rows.append(
            {
                "Position_Group": pos,
                "n": group.height,
                "intercept": model.intercept_,
                "slope": model.coef_[0],
                "r2": model.score(X, y),
                "mean_peak": float(group.get_column("Peak_Overall").mean()),
                "mean_positive_econ": float(group.get_column("positive_econ").mean()),
            }
        )
    return pl.DataFrame(rows).sort("slope", descending=True)


position_slopes = fit_position_slopes(df.filter(pl.col("is_positive_econ")))
position_slopes


In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=position_slopes.to_pandas(), x="Position_Group", y="slope")
plt.title("Position-Specific log Economic Value Per Peak Overall Point")
plt.xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()


## Residual Check: What Does Talent-Only Miss?

If a universal talent model is missing positional economics, residuals should show systematic position effects.

In [ ]:
talent_model = LinearRegression().fit(
    df.select("Peak_Overall").to_numpy(),
    df.get_column("log_positive_econ").to_numpy(),
)
pred = talent_model.predict(df.select("Peak_Overall").to_numpy())
df_resid = df.with_columns(
    pl.Series("talent_only_log_pred", pred),
    pl.Series("talent_only_residual", df.get_column("log_positive_econ").to_numpy() - pred),
)

resid_summary = (
    df_resid.group_by("Position_Group")
    .agg(
        pl.len().alias("n"),
        pl.col("talent_only_residual").mean().alias("mean_residual"),
        pl.col("talent_only_residual").median().alias("median_residual"),
    )
    .sort("mean_residual", descending=True)
)
resid_summary


In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=resid_summary.to_pandas(), x="Position_Group", y="mean_residual")
plt.axhline(0, color="black", linewidth=1)
plt.title("Mean Residual By Position After Talent-Only Economic Model")
plt.xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()


## Draft-Class Ranking Metrics

These quick metrics apply `K` within each draft class, then average across draft classes. This matches the real decision context: ranking one live draft class at a time.

In [ ]:
def ndcg_at_k(y_true, y_score, k):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    order = np.argsort(y_score)[::-1][:k]
    gains = np.maximum(y_true[order], 0)
    discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
    dcg = np.sum(gains * discounts)
    ideal = np.sort(np.maximum(y_true, 0))[::-1][:k]
    ideal_dcg = np.sum(ideal * discounts)
    return 0.0 if ideal_dcg == 0 else float(dcg / ideal_dcg)


def mean_ndcg_by_year(data: pl.DataFrame, score_col: str, target_col: str, k: int):
    scores = []
    for group in data.partition_by("Year", maintain_order=True):
        scores.append(
            ndcg_at_k(
                group.get_column(target_col).to_numpy(),
                group.get_column(score_col).to_numpy(),
                k=k,
            )
        )
    return float(np.mean(scores))


def topk_actual_value_by_year(data: pl.DataFrame, score_col: str, target_col: str, k: int):
    values = []
    for group in data.partition_by("Year", maintain_order=True):
        top = group.sort(score_col, descending=True).head(k)
        values.append(float(top.get_column(target_col).clip(lower_bound=0).sum()))
    return float(np.mean(values))


def bust_rate_by_year(data: pl.DataFrame, score_col: str, success_col: str, k: int):
    rates = []
    for group in data.partition_by("Year", maintain_order=True):
        top = group.sort(score_col, descending=True).head(k)
        rates.append(1 - float(top.get_column(success_col).cast(pl.Float64).mean()))
    return float(np.mean(rates))


## Compare Simple Board Scores

This compares a talent-only score, an economic target baseline, and simple position-weighted talent scores. The `actual_economic` score is an oracle upper-bound reference, not a usable model.

In [ ]:
score_df = df.clone()

# Mean-value weights: simple average economic value per peak point by position.
weight_map = {row[0]: row[1] for row in position_summary.select(["Position_Group", "econ_per_peak"]).iter_rows()}
slope_map = {row[0]: row[1] for row in position_slopes.select(["Position_Group", "slope"]).iter_rows()}

score_df = score_df.with_columns(
    pl.col("Peak_Overall").alias("score_peak_only"),
    (
        pl.col("Peak_Overall")
        * pl.col("Position_Group").map_elements(lambda value: weight_map.get(value), return_dtype=pl.Float64)
    ).alias("score_weighted_peak_mean"),
    (
        pl.col("Peak_Overall")
        * pl.col("Position_Group").map_elements(lambda value: slope_map.get(value), return_dtype=pl.Float64)
    ).alias("score_weighted_peak_slope"),
    pl.col("positive_econ").alias("score_actual_economic_oracle"),
)

score_cols = [
    "score_peak_only",
    "score_weighted_peak_mean",
    "score_weighted_peak_slope",
    "score_actual_economic_oracle",
]

rows = []
for score_col in score_cols:
    rows.append(
        {
            "score": score_col,
            "econ_ndcg_at_32": mean_ndcg_by_year(score_df, score_col, "positive_econ", 32),
            "econ_ndcg_at_64": mean_ndcg_by_year(score_df, score_col, "positive_econ", 64),
            "peak_ndcg_at_64": mean_ndcg_by_year(score_df, score_col, "Peak_Overall", 64),
            "games_ndcg_at_64": mean_ndcg_by_year(score_df, score_col, "Career_Games_Played", 64),
            "top64_actual_econ": topk_actual_value_by_year(score_df, score_col, "positive_econ", 64),
            "bust_rate_at_32": bust_rate_by_year(score_df, score_col, "is_positive_econ", 32),
        }
    )

score_comparison = pl.DataFrame(rows).sort("econ_ndcg_at_64", descending=True)
score_comparison


## Inspect Divergences

The most useful rows to inspect are players that talent-only and position-weighted talent rank very differently.

In [ ]:
inspect_year = int(score_df.get_column("Year").max())
year_board = score_df.filter(pl.col("Year") == inspect_year).with_columns(
    pl.col("score_peak_only").rank(method="ordinal", descending=True).alias("rank_score_peak_only"),
    pl.col("score_weighted_peak_mean").rank(method="ordinal", descending=True).alias("rank_score_weighted_peak_mean"),
    pl.col("score_weighted_peak_slope").rank(method="ordinal", descending=True).alias("rank_score_weighted_peak_slope"),
    pl.col("positive_econ").rank(method="ordinal", descending=True).alias("rank_positive_econ"),
).with_columns(
    (pl.col("rank_score_peak_only") - pl.col("rank_score_weighted_peak_mean")).alias(
        "weighted_vs_peak_rank_delta"
    )
)

cols = [
    "Player_ID",
    "Year",
    "Position_Group",
    "Position",
    "Peak_Overall",
    "positive_econ",
    "Career_Games_Played",
    "score_peak_only",
    "score_weighted_peak_mean",
    "rank_score_peak_only",
    "rank_score_weighted_peak_mean",
    "weighted_vs_peak_rank_delta",
]

year_board.sort("weighted_vs_peak_rank_delta", descending=True).select(cols).head(25)


In [ ]:
year_board.sort("weighted_vs_peak_rank_delta").select(cols).head(25)


## Next Steps

If these simple weighted-talent scores look promising:

1. Replace `Peak_Overall` with out-of-fold predictions from a talent regressor.
2. Learn position weights only on the training years, then evaluate on held-out draft years.
3. Compare against a direct economic regressor with the same per-draft-class cross-outcome metrics.
4. Inspect top-board divergences manually by draft class.

The strongest evidence for the hybrid approach would be that `predicted_talent * learned_position_weight` preserves most economic NDCG/top-k value while improving talent/longevity robustness and interpretability.

## Chronological Hold-Out Analysis (Train Weights, Test Board)

This section avoids in-sample leakage: position conversion weights are learned on older draft years only, then evaluated on later years.


In [ ]:
# Split point can be adjusted; keep enough years on each side.
split_year = int(df.get_column("Year").quantile(0.8))

train_df = df.filter(pl.col("Year") <= split_year)
test_df = df.filter(pl.col("Year") > split_year)

print("split_year:", split_year)
print(
    "train years:",
    int(train_df.get_column("Year").min()),
    "to",
    int(train_df.get_column("Year").max()),
    "n=",
    train_df.height,
)
print(
    "test years :",
    int(test_df.get_column("Year").min()),
    "to",
    int(test_df.get_column("Year").max()),
    "n=",
    test_df.height,
)


In [ ]:
# Learn slope weights only from positive-economic examples in training years.
train_pos = train_df.filter(pl.col("positive_econ") > 0)

slope_rows = []
for pos in train_pos.get_column("Position_Group").unique().to_list():
    g = train_pos.filter(pl.col("Position_Group") == pos)
    if g.height < 20:
        continue
    X = g.select("Peak_Overall").to_numpy()
    y = np.log1p(g.get_column("positive_econ").to_numpy())
    m = LinearRegression().fit(X, y)
    slope_rows.append(
        {
            "Position_Group": pos,
            "slope": float(m.coef_[0]),
            "intercept": float(m.intercept_),
            "n": g.height,
        }
    )

train_slopes = pl.DataFrame(slope_rows).sort("slope", descending=True)
train_slope_map = {row[0]: row[1] for row in train_slopes.select(["Position_Group", "slope"]).iter_rows()}
global_fallback_slope = float(train_slopes.get_column("slope").median())

train_slopes


In [ ]:
# Build scores for hold-out test years only.
holdout = test_df.with_columns(
    pl.col("Peak_Overall").alias("score_peak_only"),
    pl.col("Position_Group")
    .map_elements(lambda value: train_slope_map.get(value), return_dtype=pl.Float64)
    .fill_null(global_fallback_slope)
    .alias("position_slope_weight"),
).with_columns(
    (pl.col("Peak_Overall") * pl.col("position_slope_weight")).alias(
        "score_weighted_peak_slope_holdout"
    )
)

holdout.select(
    [
        "Position_Group",
        "Peak_Overall",
        "position_slope_weight",
        "score_peak_only",
        "score_weighted_peak_slope_holdout",
    ]
).head()


In [ ]:
# Compare hold-out ranking and value capture.
rows = []
for score_col in ["score_peak_only", "score_weighted_peak_slope_holdout"]:
    rows.append(
        {
            "score": score_col,
            "econ_ndcg_at_32": mean_ndcg_by_year(holdout, score_col, "positive_econ", 32),
            "econ_ndcg_at_64": mean_ndcg_by_year(holdout, score_col, "positive_econ", 64),
            "peak_ndcg_at_64": mean_ndcg_by_year(holdout, score_col, "Peak_Overall", 64),
            "games_ndcg_at_64": mean_ndcg_by_year(holdout, score_col, "Career_Games_Played", 64),
            "top64_actual_econ": topk_actual_value_by_year(holdout, score_col, "positive_econ", 64),
            "bust_rate_at_32": bust_rate_by_year(holdout, score_col, "is_positive_econ", 32),
        }
    )

holdout_comparison = pl.DataFrame(rows).sort("econ_ndcg_at_64", descending=True)
holdout_comparison


In [ ]:
# Year-by-year difference view: where weighted slope helps or hurts.
year_rows = []
for g in holdout.partition_by("Year", maintain_order=True):
    y = int(g.get_column("Year")[0])
    peak_ndcg = ndcg_at_k(
        g.get_column("positive_econ").to_numpy(),
        g.get_column("score_peak_only").to_numpy(),
        64,
    )
    w_ndcg = ndcg_at_k(
        g.get_column("positive_econ").to_numpy(),
        g.get_column("score_weighted_peak_slope_holdout").to_numpy(),
        64,
    )

    top_peak = (
        g.sort("score_peak_only", descending=True)
        .head(64)
        .get_column("positive_econ")
        .clip(lower_bound=0)
        .sum()
    )
    top_w = (
        g.sort("score_weighted_peak_slope_holdout", descending=True)
        .head(64)
        .get_column("positive_econ")
        .clip(lower_bound=0)
        .sum()
    )

    year_rows.append(
        {
            "Year": y,
            "econ_ndcg64_peak_only": peak_ndcg,
            "econ_ndcg64_weighted": w_ndcg,
            "delta_ndcg64_weighted_minus_peak": w_ndcg - peak_ndcg,
            "top64_econ_peak_only": float(top_peak),
            "top64_econ_weighted": float(top_w),
            "delta_top64_econ_weighted_minus_peak": float(top_w - top_peak),
        }
    )

year_deltas = pl.DataFrame(year_rows).sort("Year")
year_deltas


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

year_deltas_pd = year_deltas.to_pandas()

sns.barplot(data=year_deltas_pd, x="Year", y="delta_ndcg64_weighted_minus_peak", ax=axes[0], color="steelblue")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Hold-Out Year Delta: Econ NDCG@64 (Weighted - Peak)")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=year_deltas_pd, x="Year", y="delta_top64_econ_weighted_minus_peak", ax=axes[1], color="darkorange")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Hold-Out Year Delta: Top64 Actual Econ (Weighted - Peak)")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()


### Optional: Fixed Split Sensitivity

Repeat the hold-out check with different split years (for example 60/40, 70/30, 80/20) to ensure the conclusion is not a split artifact.


In [ ]:
def evaluate_split(split_year):
    tr = df.filter(pl.col("Year") <= split_year)
    te = df.filter(pl.col("Year") > split_year)

    tr_pos = tr.filter(pl.col("positive_econ") > 0)
    rows = []
    for pos in tr_pos.get_column("Position_Group").unique().to_list():
        g = tr_pos.filter(pl.col("Position_Group") == pos)
        if g.height < 20:
            continue
        X = g.select("Peak_Overall").to_numpy()
        y = np.log1p(g.get_column("positive_econ").to_numpy())
        m = LinearRegression().fit(X, y)
        rows.append((pos, float(m.coef_[0])))

    if not rows or te.height == 0:
        return None

    slope_map = dict(rows)
    fallback = float(np.median([v for _, v in rows]))

    te = te.with_columns(
        pl.col("Peak_Overall").alias("score_peak_only"),
        (
            pl.col("Peak_Overall")
            * pl.col("Position_Group")
            .map_elements(lambda value: slope_map.get(value), return_dtype=pl.Float64)
            .fill_null(fallback)
        ).alias("score_weighted"),
    )

    out = {
        "split_year": int(split_year),
        "n_train": int(tr.height),
        "n_test": int(te.height),
        "econ_ndcg64_peak": mean_ndcg_by_year(te, "score_peak_only", "positive_econ", 64),
        "econ_ndcg64_weighted": mean_ndcg_by_year(te, "score_weighted", "positive_econ", 64),
        "top64_econ_peak": topk_actual_value_by_year(te, "score_peak_only", "positive_econ", 64),
        "top64_econ_weighted": topk_actual_value_by_year(te, "score_weighted", "positive_econ", 64),
    }
    out["delta_ndcg64"] = out["econ_ndcg64_weighted"] - out["econ_ndcg64_peak"]
    out["delta_top64_econ"] = out["top64_econ_weighted"] - out["top64_econ_peak"]
    return out

split_candidates = sorted(df.get_column("Year").unique().to_list())
split_candidates = [
    y
    for y in split_candidates
    if y > df.get_column("Year").min() + 10 and y < df.get_column("Year").max() - 5
]
sample_splits = [split_candidates[int(len(split_candidates) * q)] for q in [0.6, 0.7, 0.8]]

sensitivity_rows = [evaluate_split(y) for y in sample_splits]
sensitivity_rows = [r for r in sensitivity_rows if r is not None]
pl.DataFrame(sensitivity_rows)
